# Preview Equirectangular Data

这个 notebook 用来快速预览单个 equirectangular `.npz` 样本里的所有 modality，并把每个 hit 单独可视化出来。

包含的内容：
- `model_rgb` 的 5 个 hit
- `model_depth` 的 5 个 hit
- `model_normal` 的 5 个 hit
- `edge_depth` 的 3 个 hit
- 每个 hit 的 valid mask 和 coverage
- 每个 depth hit 的分位数统计

使用方式：先改第 2 个代码单元里的 `sample_uid` 或 `sample_path`，再顺序运行后面的单元。

In [ ]:
%matplotlib inline

from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from edge3d_tensor_format import load_sample_modalities

plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['axes.facecolor'] = '#111111'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['image.interpolation'] = 'nearest'

sample_dir = Path('/home/devdata/edge3d_data/equirectangular_data')
sample_uid = '013755b44b7445fc8d4c864febe96c59'
sample_path = None
show_example_uids = 12

In [ ]:
def resolve_sample_path(sample_dir: Path, sample_uid: str | None = None, sample_path: str | Path | None = None) -> Path:
    if sample_path is not None:
        resolved = Path(sample_path).expanduser().resolve()
        if not resolved.exists():
            raise FileNotFoundError(f'Sample path does not exist: {resolved}')
        return resolved

    if sample_uid:
        candidate = sample_dir / f'{sample_uid}.npz'
        if candidate.exists():
            return candidate
        raise FileNotFoundError(f'UID not found under {sample_dir}: {sample_uid}')

    candidates = sorted(sample_dir.glob('*.npz'))
    if not candidates:
        raise FileNotFoundError(f'No .npz files found under {sample_dir}')
    return candidates[0]


def list_example_sample_uids(sample_dir: Path, limit: int = 12) -> list[str]:
    return [path.stem for path in sorted(sample_dir.glob('*.npz'))[:limit]]


def finite_stats(values: np.ndarray) -> dict[str, float]:
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return {
            'count': 0,
            'min': float('nan'),
            'p01': float('nan'),
            'p50': float('nan'),
            'p99': float('nan'),
            'max': float('nan'),
        }
    return {
        'count': int(finite.size),
        'min': float(np.min(finite)),
        'p01': float(np.percentile(finite, 1.0)),
        'p50': float(np.percentile(finite, 50.0)),
        'p99': float(np.percentile(finite, 99.0)),
        'max': float(np.max(finite)),
    }


def build_sample_summary(sample: dict, sample_path: Path) -> dict[str, object]:
    model_valid = np.isfinite(sample['model_depth'])
    edge_valid = np.isfinite(sample['edge_depth'])
    return {
        'sample_path': str(sample_path),
        'uid': sample['uid'],
        'resolution': int(sample['resolution']),
        'width': int(sample['width']),
        'model_rgb_shape': tuple(sample['model_rgb'].shape),
        'model_depth_shape': tuple(sample['model_depth'].shape),
        'model_normal_shape': tuple(sample['model_normal'].shape),
        'edge_depth_shape': tuple(sample['edge_depth'].shape),
        'model_hit_coverage': [float(x) for x in model_valid.mean(axis=(1, 2))],
        'edge_hit_coverage': [float(x) for x in edge_valid.mean(axis=(1, 2))],
    }


sample_path_resolved = resolve_sample_path(sample_dir=sample_dir, sample_uid=sample_uid, sample_path=sample_path)
sample = load_sample_modalities(sample_path_resolved)
summary = build_sample_summary(sample, sample_path_resolved)

display(Markdown('## Loaded Sample'))
for key, value in summary.items():
    print(f'{key}: {value}')

print('')
print('Example UIDs:')
for uid in list_example_sample_uids(sample_dir, limit=show_example_uids):
    print(' -', uid)

In [ ]:
def to_hwc(image_chw: np.ndarray) -> np.ndarray:
    return np.moveaxis(np.asarray(image_chw, dtype=np.float32), 0, -1)


def percentile_range(values: np.ndarray, lower: float = 1.0, upper: float = 99.0) -> tuple[float, float]:
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return 0.0, 1.0
    lo, hi = np.percentile(finite, [lower, upper])
    if not np.isfinite(lo) or not np.isfinite(hi):
        lo = float(np.min(finite))
        hi = float(np.max(finite))
    if hi <= lo:
        hi = lo + 1e-6
    return float(lo), float(hi)


def make_rgb_preview(rgb_hit: np.ndarray, valid_mask: np.ndarray) -> np.ndarray:
    image = to_hwc(rgb_hit)
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        preview = np.zeros_like(image)
    else:
        lo, hi = np.percentile(finite, [1.0, 99.0])
        if hi <= lo:
            hi = lo + 1e-6
        preview = np.clip((image - lo) / (hi - lo), 0.0, 1.0)
    preview = preview.copy()
    preview[~valid_mask] = 0.0
    return preview


def make_normal_preview(normal_hit: np.ndarray, valid_mask: np.ndarray) -> np.ndarray:
    image = to_hwc(normal_hit)
    preview = np.clip(0.5 * (image + 1.0), 0.0, 1.0)
    preview = preview.copy()
    preview[~valid_mask] = 0.0
    return preview


def make_mask_preview(valid_mask: np.ndarray) -> np.ndarray:
    valid = valid_mask.astype(np.float32)
    return np.stack([valid, valid, valid], axis=-1)


def make_depth_preview(depth_hit: np.ndarray, cmap_name: str = 'magma', lower: float = 1.0, upper: float = 99.0):
    lo, hi = percentile_range(depth_hit, lower=lower, upper=upper)
    safe = np.nan_to_num(depth_hit, nan=lo).astype(np.float32)
    normed = np.clip((safe - lo) / (hi - lo), 0.0, 1.0)
    rgba = plt.get_cmap(cmap_name)(normed)
    rgba = rgba.copy()
    rgba[~np.isfinite(depth_hit)] = np.array([0.08, 0.08, 0.08, 1.0], dtype=np.float32)
    return rgba, (lo, hi)


def render_depth_stats_table(sample: dict) -> None:
    lines = [
        '| branch | hit | coverage | min | p01 | p50 | p99 | max |',
        '|---|---:|---:|---:|---:|---:|---:|---:|',
    ]

    for hit_idx, depth_hit in enumerate(sample['model_depth'], start=1):
        valid_mask = np.isfinite(depth_hit)
        stats = finite_stats(depth_hit)
        lines.append(
            f"| model | {hit_idx} | {valid_mask.mean():.4f} | {stats['min']:.4f} | {stats['p01']:.4f} | {stats['p50']:.4f} | {stats['p99']:.4f} | {stats['max']:.4f} |"
        )

    for hit_idx, depth_hit in enumerate(sample['edge_depth'], start=1):
        valid_mask = np.isfinite(depth_hit)
        stats = finite_stats(depth_hit)
        lines.append(
            f"| edge | {hit_idx} | {valid_mask.mean():.4f} | {stats['min']:.4f} | {stats['p01']:.4f} | {stats['p50']:.4f} | {stats['p99']:.4f} | {stats['max']:.4f} |"
        )

    display(Markdown('\n'.join(lines)))


def plot_hit_coverages(sample: dict) -> None:
    model_cov = np.isfinite(sample['model_depth']).mean(axis=(1, 2))
    edge_cov = np.isfinite(sample['edge_depth']).mean(axis=(1, 2))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

    axes[0].bar(np.arange(1, len(model_cov) + 1), model_cov, color='#d95f02')
    axes[0].set_title('Model hit coverage')
    axes[0].set_xlabel('Hit index')
    axes[0].set_ylabel('Finite ratio')
    axes[0].set_ylim(0.0, 1.0)

    axes[1].bar(np.arange(1, len(edge_cov) + 1), edge_cov, color='#1b9e77')
    axes[1].set_title('Edge hit coverage')
    axes[1].set_xlabel('Hit index')
    axes[1].set_ylabel('Finite ratio')
    axes[1].set_ylim(0.0, 1.0)

    plt.show()


def plot_model_hits(sample: dict) -> None:
    model_rgb = sample['model_rgb']
    model_depth = sample['model_depth']
    model_normal = sample['model_normal']
    num_hits = model_depth.shape[0]

    fig, axes = plt.subplots(num_hits, 4, figsize=(22, 4 * num_hits), constrained_layout=True)
    if num_hits == 1:
        axes = np.expand_dims(axes, axis=0)

    for hit_idx in range(num_hits):
        valid_mask = np.isfinite(model_depth[hit_idx])
        rgb_preview = make_rgb_preview(model_rgb[hit_idx], valid_mask)
        normal_preview = make_normal_preview(model_normal[hit_idx], valid_mask)
        depth_preview, (lo, hi) = make_depth_preview(model_depth[hit_idx], cmap_name='magma')
        mask_preview = make_mask_preview(valid_mask)

        panels = [
            (rgb_preview, f'model_rgb hit {hit_idx + 1}'),
            (depth_preview, f'model_depth hit {hit_idx + 1}\n[{lo:.3f}, {hi:.3f}]'),
            (normal_preview, f'model_normal hit {hit_idx + 1}'),
            (mask_preview, f'valid mask hit {hit_idx + 1}\ncoverage={valid_mask.mean():.4f}'),
        ]

        for ax, (image, title) in zip(axes[hit_idx], panels):
            ax.imshow(image)
            ax.set_title(title)
            ax.axis('off')

    fig.suptitle(f"Model modalities for {sample['uid']}", fontsize=16, y=1.01)
    plt.show()


def plot_edge_hits(sample: dict) -> None:
    edge_depth = sample['edge_depth']
    num_hits = edge_depth.shape[0]

    fig, axes = plt.subplots(num_hits, 2, figsize=(14, 4 * num_hits), constrained_layout=True)
    if num_hits == 1:
        axes = np.expand_dims(axes, axis=0)

    for hit_idx in range(num_hits):
        valid_mask = np.isfinite(edge_depth[hit_idx])
        depth_preview, (lo, hi) = make_depth_preview(edge_depth[hit_idx], cmap_name='viridis')
        mask_preview = make_mask_preview(valid_mask)

        panels = [
            (depth_preview, f'edge_depth hit {hit_idx + 1}\n[{lo:.3f}, {hi:.3f}]'),
            (mask_preview, f'valid mask hit {hit_idx + 1}\ncoverage={valid_mask.mean():.4f}'),
        ]

        for ax, (image, title) in zip(axes[hit_idx], panels):
            ax.imshow(image)
            ax.set_title(title)
            ax.axis('off')

    fig.suptitle(f"Edge modalities for {sample['uid']}", fontsize=16, y=1.01)
    plt.show()

In [ ]:
display(Markdown('## Hit Coverage'))
plot_hit_coverages(sample)

In [ ]:
display(Markdown('## Depth Stats'))
render_depth_stats_table(sample)

In [ ]:
display(Markdown('## Model Hits'))
plot_model_hits(sample)

In [ ]:
display(Markdown('## Edge Hits'))
plot_edge_hits(sample)